# OW-OVD Task 1 Experimentation Notebook
Notebook này được cấu hình để chạy lại Task 1 từ đầu với tập dữ liệu IP102 (đã lọc còn 25 lớp), trong đó Task 1 học **7 lớp** đầu tiên và huấn luyện trong **5 epoch** sử dụng cấu hình mới tại `configs/custom/ip102_t1.py`.

In [ ]:
# Cell 1: Clone repository và thiết lập thư mục làm việc trên Kaggle
import os

repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository đã tồn tại. Đang cập nhật (pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Clone thư mục con mmyolo vào third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("Cloning mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("mmyolo đã tồn tại.")

In [ ]:
# Cell 2: Cài đặt toàn bộ môi trường và các thư viện cần thiết
# 1. Hạ cấp PyTorch & Torchvision xuống bản ổn định 2.4.0
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. Cài đặt MMCV từ bản build sẵn tương thích
!pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

# 3. Cài đặt các thư viện bổ trợ khác
!pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

# 4. Cài đặt MMDetection và MMYOLO
!pip install "mmdet>=3.1.0" --no-deps
!pip install --no-build-isolation --no-deps third_party/mmyolo

In [ ]:
# Cell 3: Vá lỗi giới hạn phiên bản MMCV trong mmdet và mmyolo
import importlib.util
import sys

packages = ['mmdet', 'mmyolo']
for pkg in packages:
    spec = importlib.util.find_spec(pkg)
    if spec and spec.origin:
        try:
            with open(spec.origin, 'r') as f:
                content = f.read()
            # Thay thế kiểm tra phiên bản để tránh crash
            new_content = content.replace('2.1.0', '2.3.0').replace('2.2.0', '2.3.0')
            with open(spec.origin, 'w') as f:
                f.write(new_content)
            print(f"-> Đã vá lỗi phiên bản MMCV cho {pkg} thành công.")
        except Exception as e:
            print(f"-> Lỗi khi vá {pkg}: {e}")

In [ ]:
# Cell 4: Tạo cấu hình ip102_t1.py ngay trên Kaggle
# Bước này giúp ghi đè hoặc tạo mới file cấu hình nếu bạn chưa commit lên git
import os

os.makedirs('configs/custom', exist_ok=True)
config_content = """_base_ = ('../../third_party/mmyolo/configs/yolov8/'
          'yolov8_l_syncbn_fast_8xb16-500e_coco.py')
custom_imports = dict(imports=['yolo_world'], allow_failed_imports=False)

# Suppress MMEngine's extremely verbose configuration printing at startup
import mmengine
try:
    mmengine.Config.pretty_text = property(lambda self: "[Config dump suppressed for cleaner logs]")
    from mmengine.logging import MMLogger
    _orig_info = MMLogger.info
    def _clean_info(self, msg, *args, **kwargs):
        msg_str = str(msg)
        if any(term in msg_str for term in ['OurDetector(', 'MultiModalYOLOBackbone(', 'paramwise_options', 'Checkpoints will be saved to']):
            return
        _orig_info(self, msg, *args, **kwargs)
    MMLogger.info = _clean_info
except Exception:
    pass

# Fool-proof monkey patch to fix double 'test/test/' path bug in Kaggle datasets
try:
    from mmyolo.datasets import YOLOv5CocoDataset
    _orig_parse = YOLOv5CocoDataset.parse_data_info
    def _patched_parse(self, raw_data_info):
        data_info = _orig_parse(self, raw_data_info)
        if 'img_path' in data_info and 'test/test/' in data_info['img_path']:
            data_info['img_path'] = data_info['img_path'].replace('test/test/', 'test/')
        return data_info
    YOLOv5CocoDataset.parse_data_info = _patched_parse
except Exception:
    pass

# Dynamically load class names from IP102 annotations
import json
import os
class_names = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection/train.json',
    'train.json'
]:
    if os.path.exists(path):
        try:
            with open(path, 'r') as f:
                coco_data = json.load(f)
            categories = sorted(coco_data['categories'], key=lambda x: x['id'])
            class_names = [cat['name'] for cat in categories]
            break
        except Exception:
            pass

if class_names is None:
    class_names = ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', '39', '45', '46', '47', '48', '49', '50', '51', '66', '67', '69', '70', '86', '101']

# open world setting
prev_intro_cls = 0
cur_intro_cls = 7
train_json = '/kaggle/input/datasets/nta212/ip102-for-object-detection/train.json'
test_json = '/kaggle/input/datasets/nta212/ip102-for-object-detection/test.json'
embedding_path = 'data/IP102/ip102_gt_embeddings.npy'
att_embeddings = 'data/IP102/task_att_1_embeddings.pth'
pipline = [dict(type='att_select', log_start_epoch=1)]
thr = 0.55
alpha = 0.2
use_sigmoid = True
distributions = 'data/IP102/mowod_distribution_sim1.pth'
top_k = 10

# yolo world setting
num_classes = 7
num_training_classes = 7
max_epochs = 5
close_mosaic_epochs = max_epochs
save_epoch_intervals = 1
text_channels = 512
neck_embed_channels = [128, 256, _base_.last_stage_out_channels // 2]
neck_num_heads = [4, 8, _base_.last_stage_out_channels // 2 // 32]
base_lr = 1e-4
weight_decay = 0.05
train_batch_size_per_gpu = 24
load_from = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
persistent_workers = True

# model settings
model = dict(type='OurDetector',
             mm_neck=True,
             num_train_classes=num_training_classes,
             num_test_classes=num_classes,
             embedding_path=embedding_path,
             prompt_dim=text_channels,
             num_prompts=len(class_names),
             pipline=pipline,
             data_preprocessor=dict(type='YOLOv5DetDataPreprocessor'),
             backbone=dict(_delete_=True,
                           type='MultiModalYOLOBackbone',
                           text_model=None,
                           image_model={{_base_.model.backbone}},
                           frozen_stages=4,
                           with_text_model=False),
             neck=dict(type='YOLOWorldPAFPN',
                       freeze_all=False,
                       guide_channels=text_channels,
                       embed_channels=neck_embed_channels,
                       num_heads=neck_num_heads,
                       block_cfg=dict(type='MaxSigmoidCSPLayerWithTwoConv')),
             bbox_head=dict(type='OurHead',
                            att_embeddings=att_embeddings,
                            thr=thr,
                            alpha=alpha,
                            use_sigmoid=use_sigmoid,
                            distributions=distributions,
                            prev_intro_cls=prev_intro_cls,
                            cur_intro_cls=cur_intro_cls,
                            top_k=top_k,
                            head_module=dict(
                                type='OurHeadModule',
                                freeze_all=False,
                                use_bn_head=True,
                                embed_dims=text_channels,
                                num_classes=num_training_classes,),
                            ),
             train_cfg=dict(assigner=dict(num_classes=num_training_classes)))

# dataset settings
coco_train_dataset = dict(
        _delete_=True,
        type='MultiModalDataset',
        dataset=dict(
            type='YOLOv5CocoDataset',
            metainfo=dict(classes=class_names[:7]),
            data_root='/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/JPEGImages/',
            ann_file=train_json,
            data_prefix=dict(img=''),
            filter_cfg=dict(filter_empty_gt=True, min_size=32)),
        class_text_path='data/texts/IP102/class_texts.json',
        pipeline=_base_.train_pipeline)

train_dataloader = dict(persistent_workers=persistent_workers,
                        batch_size=train_batch_size_per_gpu,
                        num_workers=4,
                        pin_memory=True,
                        collate_fn=dict(type='yolow_collate'),
                        dataset=coco_train_dataset)

custom_hooks = [
    dict(type='mmdet.PipelineSwitchHook',
         switch_epoch=max_epochs - close_mosaic_epochs,
         switch_pipeline=_base_.train_pipeline_stage2),
    dict(type='OurWorkPiplineHook'),
    dict(type='EarlyStoppingHook',
         monitor='coco/Current class AP50',
         rule='greater',
         patience=2,
         min_delta=0.001)
]

default_hooks = dict(
    checkpoint=dict(
        interval=save_epoch_intervals, max_keep_ckpts=2, save_best='coco/Current class AP50', rule='greater',
        type='CheckpointHook'),
    logger=dict(interval=50, type='LoggerHook'),
    param_scheduler=dict(
        lr_factor=0.01,
        max_epochs=max_epochs,
        scheduler_type='linear',
        type='YOLOv5ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=dict(type='mmdet.DetVisualizationHook'))

train_cfg = dict(max_epochs=max_epochs,
                 val_interval=999,
                 dynamic_intervals=[(2, 1)])

optim_wrapper = dict(
    type='AmpOptimWrapper',
    optimizer=dict(
        _delete_=True,
        type='AdamW',
        lr=base_lr,
        weight_decay=weight_decay,
        batch_size_per_gpu=train_batch_size_per_gpu),
    paramwise_cfg=dict(bias_decay_mult=0.0,
                       norm_decay_mult=0.0,
                       custom_keys={
                           'backbone.text_model':
                           dict(lr_mult=0.01),
                           'logit_scale':
                           dict(weight_decay=0.0),
                           'embeddings':
                           dict(weight_decay=0.0)
                       }),
    constructor='YOLOWv5OptimizerConstructor')

test_pipeline = [
    *_base_.test_pipeline[:-1],
    dict(type='mmdet.PackDetInputs',
         meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape',
                    'scale_factor', 'pad_param'))
]

test_dataloader = dict(
    _delete_=True,
    batch_size=24,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type='YOLOv5CocoDataset',
                 metainfo=dict(classes=class_names),
                 data_root='/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/JPEGImages/',
                 ann_file=test_json,
                 data_prefix=dict(img=''),
                 filter_cfg=dict(filter_empty_gt=True, min_size=32),
                 pipeline=test_pipeline)
)

test_evaluator = dict(_delete_=True,
                      type='OWODEvaluator',
                      prefix='coco',
                      cfg=dict(
                         dataset_root='data/IP102/voc/',
                         ann_file=test_json,
                         file_name='mowod/all_task_test.txt',
                         prev_intro_cls=prev_intro_cls,
                         cur_intro_cls=cur_intro_cls,
                         unknown_id=25,
                         class_names=class_names
                      )
                     )
val_json = '/kaggle/input/datasets/nta212/ip102-for-object-detection/val.json'

val_dataloader = dict(
    _delete_=True,
    batch_size=24,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type='YOLOv5CocoDataset',
                 metainfo=dict(classes=class_names),
                 data_root='/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/JPEGImages/',
                 ann_file=val_json,
                 data_prefix=dict(img=''),
                 filter_cfg=dict(filter_empty_gt=True, min_size=32),
                 pipeline=test_pipeline)
)

val_evaluator = dict(_delete_=True,
                      type='OWODEvaluator',
                      prefix='coco',
                      cfg=dict(
                         dataset_root='data/IP102/voc_val/',
                         ann_file=val_json,
                         file_name='mowod/all_task_val.txt',
                         prev_intro_cls=prev_intro_cls,
                         cur_intro_cls=cur_intro_cls,
                         unknown_id=25,
                         class_names=class_names
                      )
                     )
find_unused_parameters = True
"""

with open('configs/custom/ip102_t1.py', 'w', encoding='utf-8') as f:
    f.write(config_content)
print("-> Đã ghi file cấu hình configs/custom/ip102_t1.py thành công!")

In [ ]:
# Cell 5: Tạo thư mục, tải weights và sinh embeddings + XML annotations
import json
import torch
import numpy as np
import os
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Tạo thư mục
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải YOLO-World weights...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
else:
    print("-> Pretrained weights đã tồn tại.")

# 3. Đọc danh sách 25 lớp mới từ file annotation trên Kaggle
ann_path = '/kaggle/input/datasets/nta212/ip102-for-object-detection/train.json'
with open(ann_path, 'r') as f:
    coco_data = json.load(f)

categories = sorted(coco_data['categories'], key=lambda x: x['id'])
class_names = [cat['name'] for cat in categories]
num_classes = len(class_names)
print(f"-> Tổng số lớp học: {num_classes}")
print(f"-> Danh sách lớp: {class_names}")

# 4. Lưu class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 5. Sinh CLIP embeddings cho các nhãn lớp
print("-> Đang sinh class embeddings từ CLIP...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))
print("-> Đã lưu ip102_gt_embeddings.npy.")

# 6. Sinh dummy attribute embeddings
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

# 7. Sinh dummy distributions
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("-> Đã sinh attribute embeddings và distributions.")

In [ ]:
# Cell 6: Chạy huấn luyện Task 1 (Sử dụng torchrun đa GPU)
# Lưu ý: configs/custom/ip102_t1.py đã cấu hình học 7 lớp, max_epochs = 5
!PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/custom/ip102_t1.py --launcher pytorch

In [ ]:
# Cell 7: Kiểm tra và đánh giá mô hình tốt nhất sau khi huấn luyện xong
import glob
import os

# Tìm checkpoint tốt nhất
weights = glob.glob('work_dirs/ip102_t1/best_*.pth')
if weights:
    best_model_path = weights[0]
    print(f"-> Đã tìm thấy mô hình tốt nhất tại: {best_model_path}")
    # Đánh giá kiểm thử
    !PYTHONPATH=. python third_party/mmyolo/tools/test.py configs/custom/ip102_t1.py "{best_model_path}"
else:
    print("-> Không tìm thấy file weight tốt nhất. Có thể quá trình huấn luyện chưa kết thúc hoặc xảy ra lỗi!")